# CPU Thread Scaling Benchmark

Compare wall time of VMC operations (sampling w/ reuse, energy eval w/ reuse, gradient) across 1, 2, 4 CPU cores.

Uses `fPEPS_Model_reuse` with `chi = 4*D` (boundary MPS reuse), matching the `sampling_time.ipynb` setup.

Since MKL/OpenBLAS thread counts must be set before import, each thread configuration
runs in a **separate subprocess** to guarantee clean environment variable propagation.

In [4]:
import subprocess
import sys
import json
import os

# --- Configuration ---
PYTHON = '/home/sijingdu/TNVMC/VMC_code/clean_symmray/bin/python'
THREAD_COUNTS = [1, 2, 4]
Lx, Ly = 8, 8
N_f = Lx * Ly - 8
D = 10
chi = 4 * D  # chi = 4D for boundary MPS contraction
t, U = 1.0, 8.0
BATCH_SIZES = [1, 4, 8]
N_REPEATS = 3  # repeats per (thread, batch) pair for averaging

In [5]:
# Write the worker script that each subprocess will execute
WORKER_SCRIPT = r'''
import os, sys, json, time

num_threads = int(sys.argv[1])
batch_size = int(sys.argv[2])
n_repeats = int(sys.argv[3])
config_json = sys.argv[4]
config = json.loads(config_json)

# Set ALL thread env vars BEFORE any imports
for var in ['MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS', 'OMP_NUM_THREADS',
            'OPENBLAS_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS']:
    os.environ[var] = str(num_threads)

import numpy as np
import quimb.tensor as qtn
import pickle
import torch
import autoray as ar
import warnings
warnings.filterwarnings('ignore')

from vmc_torch.experiment.vmap.vmap_utils import (
    random_initial_config, sample_next_reuse, evaluate_energy_reuse, compute_grads,
)
from vmc_torch.experiment.vmap.models import fPEPS_Model_reuse
from vmc_torch.hamiltonian_torch import spinful_Fermi_Hubbard_square_lattice_torch
from vmc_torch.experiment.vmap.vmap_torch_utils import robust_svd_err_catcher_wrapper
from functools import partial

torch.set_num_threads(num_threads)
torch.set_default_device('cpu')
torch.random.manual_seed(42)

ar.register_function('torch', 'linalg.svd',
    lambda x: robust_svd_err_catcher_wrapper(x, jitter=1e-16, driver=None))

# --- Setup ---
Lx, Ly = config['Lx'], config['Ly']
N_f = config['N_f']
D, chi = config['D'], config['chi']
t_hop, U_int = config['t'], config['U']

pwd = '/home/sijingdu/TNVMC/VMC_code/vmc_torch/vmc_torch/experiment/vmap/data'
params_path = f'{pwd}/{Lx}x{Ly}/t={t_hop}_U={U_int}/N={N_f}/Z2/D={D}/'
params = pickle.load(open(params_path + 'peps_su_params_U1SU.pkl', 'rb'))
skeleton = pickle.load(open(params_path + 'peps_skeleton_U1SU.pkl', 'rb'))
peps = qtn.unpack(params, skeleton)
for ts in peps.tensors:
    ts.modify(data=ts.data.to_flat() * 4)
for site in peps.sites:
    peps[site].data._label = site
    peps[site].data.indices[-1]._linearmap = ((0, 0), (1, 0), (1, 1), (0, 1))

# Build reusable fPEPS model (chi = 4D)
model_dtype = torch.float64
fpeps_model = fPEPS_Model_reuse(
    tn=peps,
    max_bond=chi,
    dtype=model_dtype,
    contract_boundary_opts={
        'mode': 'mps',
        'canonize': True,
    },
)

H = spinful_Fermi_Hubbard_square_lattice_torch(
    Lx, Ly, t_hop, U_int, N_f, pbc=False,
    n_fermions_per_spin=(N_f // 2, N_f // 2), no_u1_symmetry=False,
)

B = batch_size
B_grad = max(1, B // 2)
get_grads = partial(compute_grads, vectorize=True, vmap_grad=True, batch_size=B_grad, verbose=False)
fxs0 = torch.stack([random_initial_config(N_f, peps.nsites) for _ in range(B)]).to(torch.long)

# Cache bMPS skeletons (needed for reuse model)
fpeps_model.cache_bMPS_skeleton(fxs0[0])

# --- Warmup (1 run, discarded) ---
with torch.no_grad():
    fxs_w, amps_w = sample_next_reuse(fxs0.clone(), fpeps_model, H.graph)
    evaluate_energy_reuse(fxs_w, fpeps_model, H, amps_w)
get_grads(fxs0.clone(), fpeps_model)

# --- Timed runs ---
sample_times = []
energy_times = []
grad_times = []

for _ in range(n_repeats):
    fxs = fxs0.clone()

    with torch.no_grad():
        t0 = time.perf_counter()
        fxs, amps = sample_next_reuse(fxs, fpeps_model, H.graph)
        t1 = time.perf_counter()
        evaluate_energy_reuse(fxs, fpeps_model, H, amps)
        t2 = time.perf_counter()

    get_grads(fxs0.clone(), fpeps_model)
    t3 = time.perf_counter()

    sample_times.append(t1 - t0)
    energy_times.append(t2 - t1)
    grad_times.append(t3 - t2)

result = {
    'num_threads': num_threads,
    'batch_size': batch_size,
    'sample_times': sample_times,
    'energy_times': energy_times,
    'grad_times': grad_times,
}
print(json.dumps(result))
'''

worker_path = os.path.join(
    '/home/sijingdu/TNVMC/VMC_code/vmc_torch/vmc_torch/experiment/vmap/_claude',
    '_thread_bench_worker.py'
)
with open(worker_path, 'w') as f:
    f.write(WORKER_SCRIPT)
print(f'Worker script written to {worker_path}')

Worker script written to /home/sijingdu/TNVMC/VMC_code/vmc_torch/vmc_torch/experiment/vmap/_claude/_thread_bench_worker.py


In [6]:
config = json.dumps({'Lx': Lx, 'Ly': Ly, 'N_f': N_f, 'D': D, 'chi': chi, 't': t, 'U': U})

all_results = []

for nt in THREAD_COUNTS:
    for bs in BATCH_SIZES:
        print(f'Running: threads={nt}, batch_size={bs} ...', end=' ', flush=True)
        proc = subprocess.run(
            [PYTHON, worker_path, str(nt), str(bs), str(N_REPEATS), config],
            capture_output=True, text=True, timeout=1200,
        )
        if proc.returncode != 0:
            print(f'FAILED')
            print(proc.stderr[-2000:] if proc.stderr else 'no stderr')
            continue
        # The last line of stdout is the JSON result
        lines = proc.stdout.strip().split('\n')
        result = json.loads(lines[-1])
        all_results.append(result)
        avg_total = (
            sum(result['sample_times']) / N_REPEATS
            + sum(result['energy_times']) / N_REPEATS
            + sum(result['grad_times']) / N_REPEATS
        )
        print(f'avg total={avg_total:.2f}s')

print(f'\nCollected {len(all_results)} results.')

Running: threads=1, batch_size=1 ... avg total=73.05s
Running: threads=1, batch_size=4 ... avg total=180.59s
Running: threads=1, batch_size=8 ... 

TimeoutExpired: Command '['/home/sijingdu/TNVMC/VMC_code/clean_symmray/bin/python', '/home/sijingdu/TNVMC/VMC_code/vmc_torch/vmc_torch/experiment/vmap/_claude/_thread_bench_worker.py', '1', '8', '3', '{"Lx": 8, "Ly": 8, "N_f": 56, "D": 10, "chi": 40, "t": 1.0, "U": 8.0}']' timed out after 1200 seconds

In [ ]:
import numpy as np

# Organize results into a structured dict:
# data[batch_size][num_threads] = {'sample': mean, 'energy': mean, 'grad': mean, 'total': mean}
data = {}
for r in all_results:
    bs = r['batch_size']
    nt = r['num_threads']
    if bs not in data:
        data[bs] = {}
    s = np.mean(r['sample_times'])
    e = np.mean(r['energy_times'])
    g = np.mean(r['grad_times'])
    data[bs][nt] = {'sample': s, 'energy': e, 'grad': g, 'total': s + e + g}

# Print table
print(f'{"Batch":>6} {"Threads":>8} {"Sample(s)":>10} {"Energy(s)":>10} {"Grad(s)":>10} {"Total(s)":>10}')
print('-' * 60)
for bs in sorted(data.keys()):
    for nt in sorted(data[bs].keys()):
        d = data[bs][nt]
        print(f'{bs:>6} {nt:>8} {d["sample"]:>10.2f} {d["energy"]:>10.2f} {d["grad"]:>10.2f} {d["total"]:>10.2f}')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
metrics = ['sample', 'energy', 'grad', 'total']
titles = ['Sampling Sweep (reuse)', 'Energy Eval (reuse)', 'Gradient (vmap)', 'Total']

for ax, metric, title in zip(axes, metrics, titles):
    for bs in sorted(data.keys()):
        threads = sorted(data[bs].keys())
        vals = [data[bs][nt][metric] for nt in threads]
        ax.plot(threads, vals, marker='o', label=f'B={bs}')
    ax.set_xlabel('Number of CPU Threads')
    ax.set_ylabel('Wall Time (s)')
    ax.set_title(title)
    ax.set_xticks(THREAD_COUNTS)
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle(
    f'Thread Scaling (fPEPS_Model_reuse): {Lx}x{Ly}, N_f={N_f}, D={D}, $\\chi$={chi}',
    fontsize=14, y=1.02
)
plt.tight_layout()
plt.savefig(
    f'/home/sijingdu/TNVMC/VMC_code/vmc_torch/vmc_torch/experiment/vmap/_claude/thread_scaling_{Lx}x{Ly}_D{D}_chi{chi}.pdf',
    dpi=300, bbox_inches='tight'
)
plt.show()

In [ ]:
# Speedup relative to 1 thread
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, metric, title in zip(axes, metrics, titles):
    for bs in sorted(data.keys()):
        if 1 not in data[bs]:
            continue
        baseline = data[bs][1][metric]
        threads = sorted(data[bs].keys())
        speedups = [baseline / data[bs][nt][metric] for nt in threads]
        ax.plot(threads, speedups, marker='s', label=f'B={bs}')
    ax.plot(THREAD_COUNTS, THREAD_COUNTS, 'k--', alpha=0.4, label='Ideal')
    ax.set_xlabel('Number of CPU Threads')
    ax.set_ylabel('Speedup vs 1 Thread')
    ax.set_title(title)
    ax.set_xticks(THREAD_COUNTS)
    ax.legend()
    ax.grid(True, alpha=0.3)

fig.suptitle(
    f'Thread Speedup (fPEPS_Model_reuse): {Lx}x{Ly}, N_f={N_f}, D={D}, $\\chi$={chi}',
    fontsize=14, y=1.02
)
plt.tight_layout()
plt.savefig(
    f'/home/sijingdu/TNVMC/VMC_code/vmc_torch/vmc_torch/experiment/vmap/_claude/thread_speedup_{Lx}x{Ly}_D{D}_chi{chi}.pdf',
    dpi=300, bbox_inches='tight'
)
plt.show()